In [1]:
# This code extracts X% of the sampled dataset to form a validation set.
# The validation set is chosen by a random selection among the samples

In [1]:
# Cell 1 — Setup
from ase.io.trajectory import Trajectory
from ase.io import write
import numpy as np
import json, math, os, time
from pathlib import Path

In [2]:
METHOD  = "FPS"               # "FPS", "Random", "Birch" ...

In [3]:
# sampled training set

# --- User inputs ---
SRC_TRAJ = f"selected/{METHOD}_selected.traj"         # path to your source .traj
PERCENT  = 20                   # e.g. 10 for 10%
SEED     = 42                   # set None for non-deterministic
# output directory
OUT_DIR  = f"selected/valset"  # output directory
if not os.path.exists(OUT_DIR):
    os.makedirs(OUT_DIR)
OUT_PREFIX = f"{OUT_DIR}/{METHOD}_selected_valset_{SEED}_{PERCENT}"           # output stem

# Outputs
OUT_TRAJ = f"{OUT_PREFIX}.traj"
OUT_XYZ  = f"{OUT_PREFIX}.xyz"
OUT_TXT  = f"{OUT_PREFIX}.txt"

# Sanity
assert 0 < PERCENT <= 100, "PERCENT must be in (0, 100]."
assert os.path.exists(SRC_TRAJ), f"Missing: {SRC_TRAJ}"

# Cell 2 — Inspect source trajectory and choose frames
traj = Trajectory(SRC_TRAJ, mode="r")
n_frames_source = len(traj)
print(f"Source trajectory: {SRC_TRAJ}")
print(f"Number of frames in source: {n_frames_source}")

all_frame = False
if all_frame:
    n_frames = len(traj)
else:
    print("a custom number of frames will be randomly chosen from the source trajectory.")
    n_chosen = input("Number of frames to inspect (0=all)? ")
    n_frames = int(n_chosen) if n_chosen.isdigit() and int(n_chosen) > 0 else len(traj)
    if n_frames > len(traj):
        raise ValueError(f"Requested {n_frames} frames, but source only has {len(traj)} frames.")
    # Randomly select n_frames from the original trajectory
    rng = np.random.default_rng(SEED)
    frame_indices = np.sort(rng.choice(len(traj), size=n_frames, replace=False))
    traj = [traj[i] for i in frame_indices]
    print(f"Selected {n_frames} frames from the source trajectory.")

k = max(1, int(round(PERCENT * n_frames / 100.0)))

rng = np.random.default_rng(SEED)
indices = np.sort(rng.choice(n_frames, size=k, replace=False))

print(f"Frames that will be used: {n_frames}")
print(f"Sampling: {k} frames ({PERCENT}%)")
print("First 10 sampled indices:", indices[:10])

# Cell 3 — Write selected frames to .traj and .xyz
# .traj: streamed write (no big memory spike)
if os.path.exists(OUT_TRAJ):
    os.remove(OUT_TRAJ)

with Trajectory(OUT_TRAJ, mode="w") as T:
    for i in indices:
        T.write(traj[i])

# .xyz: append mode to avoid keeping all frames in RAM
if os.path.exists(OUT_XYZ):
    os.remove(OUT_XYZ)

first = True
for i in indices:
    write(OUT_XYZ, traj[i], append=not first, format="extxyz")
    first = False

print(f"Wrote: {OUT_TRAJ}")
print(f"Wrote: {OUT_XYZ}")

# Cell 4 — Manifest (.txt) with details for each selected frame
# Captures index, natoms, cell (a,b,c), pbc, energy

ts = time.strftime("%Y-%m-%d %H:%M:%S")

with open(OUT_TXT, "w", encoding="utf-8") as f:
    f.write("# Validation set manifest\n")
    f.write(f"# Source: {Path(SRC_TRAJ).resolve()}\n")
    f.write(f"# Created: {ts}\n")
    f.write(f"# Total frames: {n_frames}\n")
    f.write(f"# Percent: {PERCENT}\n")
    f.write(f"# Seed: {SEED}\n")
    f.write(f"# Selected frames: {k}\n\n")
    f.write("index\tnatoms\tcell_a\tcell_b\tcell_c\tpbc\tenergy\tforces_json\tinfo_json\n")

    for i in indices:
        atoms = traj[i]
        cell = atoms.cell.lengths()
        pbc = tuple(bool(x) for x in atoms.pbc)

        # Energy: ASE convention is atoms.info["energy"] (may vary)
        #energy = atoms.info.get("energy", "NA")
        energy = atoms.get_potential_energy()

        line = (
            f"{i}\t{len(atoms)}\t"
            f"{cell[0]:.6f}\t{cell[1]:.6f}\t{cell[2]:.6f}\t"
            f"{pbc}\t{energy}\t\n"
        )
        f.write(line)

print(f"Wrote: {OUT_TXT}")

Source trajectory: selected/FPS_selected.traj
Number of frames in source: 6671
a custom number of frames will be randomly chosen from the source trajectory.
Selected 6671 frames from the source trajectory.
Frames that will be used: 6671
Sampling: 1334 frames (20%)
First 10 sampled indices: [27 29 30 33 37 40 41 47 53 55]
Wrote: selected/valset/FPS_selected_valset_42_20.traj
Wrote: selected/valset/FPS_selected_valset_42_20.xyz
Wrote: selected/valset/FPS_selected_valset_42_20.txt


In [4]:
# %%
# Cell 5 — Write training set (complement of validation set)
TRAIN_DIR = "selected/trainset"
os.makedirs(TRAIN_DIR, exist_ok=True)
TRAIN_PREFIX = f"{TRAIN_DIR}/{METHOD}_selected_trainset_{SEED}_{100-PERCENT}"

TRAIN_TRAJ = f"{TRAIN_PREFIX}.traj"
TRAIN_XYZ  = f"{TRAIN_PREFIX}.xyz"
TRAIN_TXT  = f"{TRAIN_PREFIX}.txt"

# Build complement indices
all_indices = np.arange(n_frames)
train_indices = np.setdiff1d(all_indices, indices)

print(f"Training frames: {len(train_indices)} ({100-PERCENT}%)")

# .traj
if os.path.exists(TRAIN_TRAJ):
    os.remove(TRAIN_TRAJ)

with Trajectory(TRAIN_TRAJ, mode="w") as T:
    for i in train_indices:
        T.write(traj[i])

# .xyz
if os.path.exists(TRAIN_XYZ):
    os.remove(TRAIN_XYZ)

first = True
for i in train_indices:
    write(TRAIN_XYZ, traj[i], append=not first, format="extxyz")
    first = False

# .txt manifest
ts = time.strftime("%Y-%m-%d %H:%M:%S")
with open(TRAIN_TXT, "w", encoding="utf-8") as f:
    f.write("# Training set manifest\n")
    f.write(f"# Source: {Path(SRC_TRAJ).resolve()}\n")
    f.write(f"# Created: {ts}\n")
    f.write(f"# Total frames: {n_frames}\n")
    f.write(f"# Percent kept: {100-PERCENT}\n")
    f.write(f"# Seed: {SEED}\n")
    f.write(f"# Training frames: {len(train_indices)}\n\n")
    f.write("index\tnatoms\tcell_a\tcell_b\tcell_c\tpbc\tenergy\n")

    for i in train_indices:
        atoms = traj[i]
        cell = atoms.cell.lengths()
        pbc = tuple(bool(x) for x in atoms.pbc)
        energy = atoms.get_potential_energy()
        line = (
            f"{i}\t{len(atoms)}\t"
            f"{cell[0]:.6f}\t{cell[1]:.6f}\t{cell[2]:.6f}\t"
            f"{pbc}\t{energy}\n"
        )
        f.write(line)

print(f"Wrote: {TRAIN_TRAJ}")
print(f"Wrote: {TRAIN_XYZ}")
print(f"Wrote: {TRAIN_TXT}")

Training frames: 5337 (80%)
Wrote: selected/trainset/FPS_selected_trainset_42_80.traj
Wrote: selected/trainset/FPS_selected_trainset_42_80.xyz
Wrote: selected/trainset/FPS_selected_trainset_42_80.txt
